In [3]:
import os
import glob
import re
import pandas as pd
import plotly.graph_objects as go

In [4]:

# ==========================================================
# PATHS
# ==========================================================
BASE_CSV_DIR  = r"./../../dataFolders/MuscaChasingBeads/Global2Local"
BASE_PLOT_DIR = r"./../../dataFolders/MuscaChasingBeads/Figures/Global2Local"

# Define Skeleton/Labels
LABELS = ['Head','L_WB','R_WB','L_WT','R_WT','Abdomen']
SKELETON = [(0,5),(0,1),(0,2),(1,2),(1,3),(2,4),(5,1),(5,2)]

# ==========================================================
# HELPERS
# ==========================================================
def add_animation_controls(fig, frames):
    """Adds the Play/Pause buttons and the Frame Slider toggle."""
    fig.update_layout(
        updatemenus=[dict(
            type="buttons",
            showactive=False,
            x=0.05, y=1.15,
            buttons=[
                dict(label="Play", method="animate",
                     args=[None, {"frame": {"duration": 30, "redraw": True}, "fromcurrent": True}]),
                dict(label="Pause", method="animate",
                     args=[[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate"}])
            ]
        )],
        sliders=[dict(
            active=0, x=0.1, y=0.05, len=0.85,
            currentvalue=dict(prefix="Frame: "),
            steps=[dict(method="animate", label=str(f),
                        args=[[str(f)], {"mode": "immediate", "frame": {"duration": 0, "redraw": True}}])
                   for f in frames]
        )]
    )

# ==========================================================
# PLOTTER
# ==========================================================
def create_3d_animation(csv_path, output_path, title, color):
    df = pd.read_csv(csv_path)
    frames_list = df["frame"].values
    
    # Drop columns that aren't coordinates to reshape safely
    coord_df = df.drop(columns=["frame"])
    pts_data = coord_df.values.reshape(len(df), 6, 3)

    anim_frames = []
    for i in range(len(df)):
        traces = []
        # Skeleton lines
        for a, b in SKELETON:
            traces.append(go.Scatter3d(
                x=[pts_data[i,a,0], pts_data[i,b,0]], 
                y=[pts_data[i,a,1], pts_data[i,b,1]], 
                z=[pts_data[i,a,2], pts_data[i,b,2]],
                mode="lines", line=dict(color="gray", width=4), showlegend=False
            ))
        # Markers
        traces.append(go.Scatter3d(
            x=pts_data[i,:,0], y=pts_data[i,:,1], z=pts_data[i,:,2],
            mode="markers+text", text=LABELS, marker=dict(size=5, color=color), showlegend=False
        ))
        anim_frames.append(go.Frame(data=traces, name=str(frames_list[i])))

    fig = go.Figure(data=anim_frames[0].data, frames=anim_frames)
    
    # Apply Aspect Ratio and Title
    fig.update_layout(
        title=title, 
        scene=dict(aspectmode="data"),
        margin=dict(l=0, r=0, b=0, t=50)
    )
    
    # ATTACH THE TOGGLE/CONTROLS
    add_animation_controls(fig, frames_list)
    
    fig.write_html(output_path)

# ==========================================================
# MAIN LOOP
# ==========================================================
local_files = sorted(glob.glob(os.path.join(BASE_CSV_DIR, "local", "*.csv")))

for path in local_files:
    # Handle filename parsing
    fname = os.path.basename(path)
    trial_name = fname.replace("_local.csv", "")
    
    save_dir = os.path.join(BASE_PLOT_DIR, trial_name)
    os.makedirs(save_dir, exist_ok=True)
    
    print(f"Generating Interactive 3D Views for: {trial_name}")
    
    # Plot Local
    create_3d_animation(path, os.path.join(save_dir, "local_view.html"), f"{trial_name} Local", "orange")
    
    # Plot Global
    global_path = path.replace("local", "global").replace("_local", "_global")
    if os.path.exists(global_path):
        create_3d_animation(global_path, os.path.join(save_dir, "global_view.html"), f"{trial_name} Global", "cyan")

print(f"\nSuccess! Animations with toggles saved to: {BASE_PLOT_DIR}")

Generating Interactive 3D Views for: Trial2_180rpm
Generating Interactive 3D Views for: Trial2_200rpm
Generating Interactive 3D Views for: Trial3_180rpm
Generating Interactive 3D Views for: Trial4_400rpm
Generating Interactive 3D Views for: Trial5_180rpm
Generating Interactive 3D Views for: Trial5_400rpm
Generating Interactive 3D Views for: Trial7_400rpm

Success! Animations with toggles saved to: ./../../dataFolders/MuscaChasingBeads/Figures/Global2Local
